## Imports and device

In [1]:
import os
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

#for other users, replace the following path with the path to your botanical images folder
image_folder = r'/Users/giacomobonanni/Desktop/botanical_orig'
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: mps


## Labeling

In [ ]:
print("Loading BLIP model and processor")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"found {len(image_files)} images to process")
for img_name in image_files:
    img_path = os.path.join(image_folder, img_name)
    base_name = os.path.splitext(img_name)[0]
    txt_path = os.path.join(image_folder, f"{base_name}.txt")
    if os.path.exists(txt_path):
        print(f"skipping {img_name}, caption already exists.")
        continue
    try:
        image = Image.open(img_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
        caption = processor.decode(outputs[0], skip_special_tokens=True)
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(caption)
        print(f"generated caption for {img_name}: {caption}")
    except Exception as e:
        print(f"error processing {img_name}: {e}")
print("Captioning process completed successfully")